# 📘 Notebook 14: Sorting

### Course: *From Zero to Data Structures & Algorithms in Python*
**Instructor:** Ioannis Tsioulis

*Module C: Algorithms & Complexity · Notebook 4 of 4 in this module · (14 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why sorting is one of the central problems of computing
- Implement **bubble sort**, **selection sort** and **insertion sort**, and explain why each is O(n²)
- Implement **merge sort** and explain why it is O(n log n)
- Compare the four algorithms on the clock and plot the result
- Use Python's built-in `sorted()` and `.sort()`, including the `key` and `reverse` options
- Choose a sensible sorting approach for a given task

> **Prerequisites:** Notebooks 11 (Big O), 12 (recursion) and 13 (searching).
>
> 🔭 **Why study sorting when Python has `sort()`?** For the same reason a medical student studies anatomy although nobody will ask them to build a human. Sorting algorithms are the classic training ground of computer science: they are small enough to understand completely, and they show, more clearly than anything else, how **a better idea beats a faster computer**.

## 1. Why sorting matters

Sorted data is useful data. Binary search needs it. Finding the largest, the smallest, the median or the duplicates becomes trivial once a list is in order. Rankings, timelines, leaderboards and every "sort by price" button depend on it.

The problem is simple to state: *rearrange the elements of a list so that each is no greater than the next.* We will solve it four times, each time with a different idea.

All four functions below sort the list **in place**: they rearrange the elements of the list they are given. They rely on one small operation, the **swap**, which Python writes in a single line.

In [ ]:
numbers = [10, 20, 30]

numbers[0], numbers[2] = numbers[2], numbers[0]     # swap the first and the last
print(numbers)

## 2. Bubble sort

### Intuition first
Walk along the list comparing each pair of **neighbours**. Whenever two neighbours are in the wrong order, swap them. After one complete walk, the largest element has drifted all the way to the end, like a bubble rising in water. Walk again, and the second largest settles next to it. Repeat until everything is in place.

### One pass, traced by hand
```
[5, 2, 8, 1]     compare 5 and 2 -> swap
[2, 5, 8, 1]     compare 5 and 8 -> leave
[2, 5, 8, 1]     compare 8 and 1 -> swap
[2, 5, 1, 8]     the largest value, 8, is now in its final place
```

In [ ]:
def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        # after i passes, the last i elements are already in place
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]

numbers = [5, 2, 8, 1, 9, 3]
bubble_sort(numbers)
print(numbers)

### Watching it work

In [ ]:
def bubble_sort_traced(arr):
    n = len(arr)
    for i in range(n - 1):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
        print(f"After pass {i + 1}: {arr}")

bubble_sort_traced([5, 2, 8, 1, 9, 3])

### How much work is it?
There is a loop inside a loop. The inner loop runs `n - 1` times, then `n - 2`, and so on down to 1. The total is `n(n - 1) / 2` comparisons, the pattern we met in an exercise of Notebook 11. Bubble sort is **O(n²)**.

## 3. Selection sort

### Intuition first
Find the **smallest** element of the list and put it first. Then find the smallest of what remains and put it second. Continue until nothing remains. This is how most people sort a hand of cards laid out on a table.

"Find the smallest" is the algorithm from Notebook 03, used here once for every position.

In [ ]:
def selection_sort(arr):
    n = len(arr)
    for i in range(n):
        min_index = i                       # assume the first unsorted element is the smallest
        for j in range(i + 1, n):
            if arr[j] < arr[min_index]:
                min_index = j               # found a smaller one: remember where
        arr[i], arr[min_index] = arr[min_index], arr[i]     # put it in position i

numbers = [5, 2, 8, 1, 9, 3]
selection_sort(numbers)
print(numbers)

The structure is again a loop inside a loop, with the same `n(n - 1) / 2` comparisons: **O(n²)**. The difference from bubble sort is in the swaps. Selection sort makes at most **one swap per pass**, while bubble sort may swap at almost every comparison. That makes selection sort faster in practice, although both have the same Big O.

## 4. Insertion sort

### Intuition first
This is how you sort cards **as you pick them up**. The cards already in your hand are sorted. You take the next card and slide it left, past every card larger than it, until it sits in the right place.

### Formal description
The left part of the list is always sorted. Take the first element of the unsorted part (the **key**), shift the larger sorted elements one place to the right, and drop the key into the gap.

In [ ]:
def insertion_sort(arr):
    for i in range(1, len(arr)):
        key = arr[i]                    # the element to insert
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]         # shift the larger element to the right
            j -= 1
        arr[j + 1] = key                # drop the key into the gap

numbers = [5, 2, 8, 1, 9, 3]
insertion_sort(numbers)
print(numbers)

In the worst case, a list in reverse order, every key travels all the way to the front: **O(n²)** again. But insertion sort has a remarkable property. If the list is **already sorted, or nearly so**, the `while` loop stops almost immediately and the whole algorithm takes only **O(n)**. On nearly-sorted data it is hard to beat.

## 5. Merge sort: a different idea

The three algorithms above have the same weakness: they compare each element with many others, one at a time. To do fundamentally better we need a fundamentally different idea.

### Intuition first
Two observations:

1. A list with zero or one element is already sorted.
2. **Merging** two lists that are already sorted into one sorted list is easy and fast: repeatedly take the smaller of the two front elements.

So: **split** the list in half, sort each half, and **merge** the two sorted halves. And how do we sort each half? In the same way. This is recursion, and the strategy is called **divide and conquer**.

### Step 1: merging two sorted lists
Each list has a finger pointing at its first unused element. We compare the two elements under the fingers, take the smaller, and advance that finger.

In [ ]:
def merge(left, right):
    result = []
    i = 0          # finger on the left list
    j = 0          # finger on the right list

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    # one of the lists is finished: take whatever remains of the other
    result.extend(left[i:])
    result.extend(right[j:])
    return result

print(merge([1, 4, 9], [2, 3, 10, 12]))

Every element is handled exactly once, so merging lists with `n` elements in total costs **O(n)**.

### Step 2: the recursive sort
With `merge` available, the sort itself is five lines.

In [ ]:
def merge_sort(arr):
    if len(arr) <= 1:                     # base case: already sorted
        return arr
    mid = len(arr) // 2
    left_half = merge_sort(arr[:mid])     # sort the left half
    right_half = merge_sort(arr[mid:])    # sort the right half
    return merge(left_half, right_half)   # merge the two sorted halves

print(merge_sort([5, 2, 8, 1, 9, 3]))

Unlike the previous three, this version **returns a new sorted list** and leaves the original unchanged.

### Why it is O(n log n)
```
[5, 2, 8, 1, 9, 3, 7, 4]                      level 0:  1 list  of 8
[5, 2, 8, 1]        [9, 3, 7, 4]              level 1:  2 lists of 4
[5, 2]   [8, 1]     [9, 3]   [7, 4]           level 2:  4 lists of 2
[5] [2]  [8] [1]    [9] [3]  [7] [4]          level 3:  8 lists of 1
```
- Halving a list of `n` elements until the pieces have one element takes **log n** levels.
- At every level, all the merging together handles each of the `n` elements once: **n** steps per level.

`n` steps per level, times `log n` levels: **O(n log n)**. For a million elements that is about 20 million steps, instead of the million million of the quadratic algorithms.

## 6. The race

Theory says merge sort should win easily on large lists. Let us check. Each algorithm sorts a **copy** of the same random list, so that the comparison is fair.

In [ ]:
import time
import random

def time_sort(sort_function, data):
    copy = list(data)                  # every algorithm gets the same input
    start = time.perf_counter()
    sort_function(copy)
    return time.perf_counter() - start

sizes = [250, 500, 1_000, 2_000, 4_000]
results = {"Bubble sort": [], "Selection sort": [], "Insertion sort": [], "Merge sort": []}

print(f"{'n':>6} {'bubble':>10} {'selection':>10} {'insertion':>10} {'merge':>10}")
for n in sizes:
    data = [random.randint(0, 100_000) for i in range(n)]
    results["Bubble sort"].append(time_sort(bubble_sort, data))
    results["Selection sort"].append(time_sort(selection_sort, data))
    results["Insertion sort"].append(time_sort(insertion_sort, data))
    results["Merge sort"].append(time_sort(merge_sort, data))
    print(f"{n:>6,} {results['Bubble sort'][-1]:>10.4f} {results['Selection sort'][-1]:>10.4f} "
          f"{results['Insertion sort'][-1]:>10.4f} {results['Merge sort'][-1]:>10.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4.5))
for name, times in results.items():
    plt.plot(sizes, times, marker="o", label=name)
plt.title("Sorting algorithm performance")
plt.xlabel("Input size (n)")
plt.ylabel("Time (seconds)")
plt.legend()
plt.grid(True)
plt.show()

### Observed result
The three quadratic algorithms curve upwards: each doubling of `n` multiplies their time by about four. Merge sort hugs the bottom of the chart. Among the quadratic ones, bubble sort is the slowest, because of all its swaps.

The gap widens quickly. In an earlier run of this experiment on larger lists, sorting **50,000** elements took about **180 seconds** with bubble sort, **71 seconds** with selection sort, and **0.2 seconds** with merge sort.

## 7. Sorting in real programs

In practice you will use Python's built-in sort. It is an O(n log n) algorithm called **Timsort**, a carefully engineered combination of merge sort and insertion sort, the two ideas you have just studied.

- `sorted(items)` returns a **new** sorted list and leaves the original alone.
- `items.sort()` sorts the list **in place** and returns nothing.

In [ ]:
numbers = [5, 2, 8, 1, 9, 3]

print(sorted(numbers))                    # a new list
print(numbers)                            # the original is unchanged

numbers.sort()                            # in place
print(numbers)

print(sorted(numbers, reverse=True))      # descending order

### Sorting by a key
Often we want to sort by something other than the natural order: words by their length, students by their grade. The `key` option takes a **function** that is applied to each element to produce the value to sort by.

In [ ]:
words = ["banana", "fig", "cherry", "kiwi"]
print(sorted(words))                  # alphabetical
print(sorted(words, key=len))         # by length

students = [
    {"name": "Anna", "grade": 8},
    {"name": "Kostas", "grade": 6},
    {"name": "Eleni", "grade": 9},
]

def get_grade(student):
    return student["grade"]

for student in sorted(students, key=get_grade, reverse=True):
    print(student["name"], student["grade"])

> ⚠️ **Common pitfalls**
>
> - `numbers = numbers.sort()` destroys your list: `.sort()` returns `None`. Write either `numbers.sort()` or `numbers = sorted(numbers)`.
> - Sorting numbers stored as text: `sorted(["10", "9", "100"])` gives `['10', '100', '9']`, because text is compared character by character.

### Summary of the four algorithms

| Algorithm | Idea | Worst case | Already sorted | Notes |
|---|---|---|---|---|
| Bubble sort | swap neighbours that are out of order | O(n²) | O(n²), or O(n) with an early exit | simplest to understand, slowest in practice |
| Selection sort | repeatedly select the smallest | O(n²) | O(n²) | very few swaps |
| Insertion sort | insert each element into the sorted part | O(n²) | **O(n)** | excellent for small or nearly-sorted lists |
| Merge sort | split, sort the halves, merge | **O(n log n)** | O(n log n) | fast and predictable; uses extra memory |

---
## ✏️ Exercises

### Exercise 1 (Trace by hand)
On paper, write the list `[4, 1, 3, 2]` after each pass of bubble sort. Then check with `bubble_sort_traced`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
# After pass 1: [1, 3, 2, 4]
# After pass 2: [1, 2, 3, 4]
# After pass 3: [1, 2, 3, 4]
bubble_sort_traced([4, 1, 3, 2])
```
</details>

### Exercise 2 (Descending order)
Write `bubble_sort_descending(arr)` that sorts a list from the largest to the smallest. Only one character of the original needs to change.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def bubble_sort_descending(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] < arr[j + 1]:          # '<' instead of '>'
                arr[j], arr[j + 1] = arr[j + 1], arr[j]

numbers = [5, 2, 8, 1, 9, 3]
bubble_sort_descending(numbers)
print(numbers)
```
</details>

### Exercise 3 (Bubble sort with an early exit)
If a complete pass makes **no swap**, the list is already sorted and there is no reason to continue. Write `bubble_sort_early(arr)` that stops as soon as this happens, and returns the number of passes it made. Test it on a list that is already sorted and on one that is not.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def bubble_sort_early(arr):
    n = len(arr)
    passes = 0
    for i in range(n):
        swapped = False
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True
        passes += 1
        if not swapped:
            break
    return passes

print(bubble_sort_early([1, 2, 3, 4, 5, 6]))     # 1 pass
print(bubble_sort_early([6, 5, 4, 3, 2, 1]))     # many passes
```

*`swapped` is a flag, the pattern from Notebook 03. With it, bubble sort takes only O(n) on a list that is already sorted.*
</details>

### Exercise 4 (Count the swaps)
Modify bubble sort and selection sort so that each **returns the number of swaps** it performed. Run both on the same list of 200 random numbers and compare.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
import random

def bubble_sort_swaps(arr):
    swaps = 0
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swaps += 1
    return swaps

def selection_sort_swaps(arr):
    swaps = 0
    n = len(arr)
    for i in range(n):
        min_index = i
        for j in range(i + 1, n):
            if arr[j] < arr[min_index]:
                min_index = j
        if min_index != i:
            arr[i], arr[min_index] = arr[min_index], arr[i]
            swaps += 1
    return swaps

data = [random.randint(0, 1000) for i in range(200)]
print("Bubble sort swaps:   ", bubble_sort_swaps(list(data)))
print("Selection sort swaps:", selection_sort_swaps(list(data)))
```

*Bubble sort makes thousands of swaps, selection sort fewer than 200. Same Big O, very different amount of real work.*
</details>

### Exercise 5 (Is it sorted?)
Write a function `is_sorted(arr)` that returns `True` if the list is in ascending order. What is its Big O? Use it to check the output of your own `merge_sort` on 1,000 random numbers.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
import random

def is_sorted(arr):
    for i in range(len(arr) - 1):
        if arr[i] > arr[i + 1]:
            return False
    return True

data = [random.randint(0, 10_000) for i in range(1000)]
print(is_sorted(data))
print(is_sorted(merge_sort(data)))
```

*One loop: O(n). Checking that a list is sorted is much cheaper than sorting it. A function like this is exactly what you write to **test** a sorting algorithm.*
</details>

### Exercise 6 (Sort the tasks)
Sort the tasks below so that the `high` priority ones come first, then `medium`, then `low`, using `sorted` with a `key` function. (Hint: a dictionary such as `{"high": 0, "medium": 1, "low": 2}` turns each priority into a number.)

In [ ]:
tasks = [
    {"name": "Buy milk", "priority": "low"},
    {"name": "Submit assignment", "priority": "high"},
    {"name": "Call Maria", "priority": "medium"},
    {"name": "Pay rent", "priority": "high"},
]
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
tasks = [
    {"name": "Buy milk", "priority": "low"},
    {"name": "Submit assignment", "priority": "high"},
    {"name": "Call Maria", "priority": "medium"},
    {"name": "Pay rent", "priority": "high"},
]

order = {"high": 0, "medium": 1, "low": 2}

def priority_rank(task):
    return order[task["priority"]]

for task in sorted(tasks, key=priority_rank):
    print(f"{task['priority']:<7} {task['name']}")
```

*The two `high` tasks keep their original relative order. A sort with this property is called **stable**, and Python's sort is stable.*
</details>

### Exercise 7 (Nearly sorted data)
Create a sorted list of 2,000 numbers and swap just two of its elements. Time `insertion_sort` and `selection_sort` on copies of it, using `time_sort`. Explain the result.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
data = list(range(2_000))
data[100], data[1500] = data[1500], data[100]      # disturb it slightly

print(f"Insertion sort: {time_sort(insertion_sort, data):.4f} seconds")
print(f"Selection sort: {time_sort(selection_sort, data):.4f} seconds")
```

*Insertion sort is dramatically faster here: almost every key is already in place, so its inner loop stops at once. Selection sort always scans the whole unsorted part, whatever the data looks like.*
</details>

### Exercise 8 (The second largest, a little harder)
Write a function `second_largest(numbers)` that returns the second largest **distinct** value of a list, in two ways: (1) by sorting, (2) with a single loop and no sorting. What is the Big O of each?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def second_largest_sorting(numbers):
    distinct = sorted(set(numbers))
    return distinct[-2]

def second_largest_loop(numbers):
    largest = None
    second = None
    for num in numbers:
        if largest is None or num > largest:
            second = largest
            largest = num
        elif num != largest and (second is None or num > second):
            second = num
    return second

data = [7, 3, 9, 9, 4, 8]
print(second_largest_sorting(data))
print(second_largest_loop(data))
```

*The sorting version is O(n log n) and takes two lines. The loop version is O(n) and needs careful thought. Sorting is often the easy route to a correct answer, and a single pass is often the faster one.*
</details>

## 🔑 Key takeaways

- **Bubble sort** swaps neighbours, **selection sort** repeatedly selects the smallest, **insertion sort** inserts each element into a sorted part. All three are **O(n²)**.
- Insertion sort is **O(n)** on data that is already nearly sorted.
- **Merge sort** splits, sorts the halves recursively and merges them: **O(n log n)**. This is **divide and conquer**.
- The same Big O can hide real differences: bubble sort makes far more swaps than selection sort.
- In real programs use `sorted()` or `.sort()`, with `key` and `reverse` when needed.
- A better algorithm beats a faster computer.

### 🏁 You have completed Module C: Algorithms & Complexity!

You can now measure how a program scales, read its Big O from its loops, write recursive functions, and you have built the classic searching and sorting algorithms by hand.

Throughout this module the data lived in ordinary Python lists. But we saw hints that the *container* matters as much as the algorithm: a set answered in an instant what a list needed a long search for, and inserting at the front of a list turned out to be surprisingly costly.

**Next:** *Module D: Data Structures*, beginning with **Stacks & Queues** (Notebook 15), where we start building containers of our own.

---
*End of Module C.*

---
*© Ioannis Tsioulis. *From Zero to Data Structures & Algorithms in Python*. Prepared for educational use.*